In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace, HuggingFaceEmbeddings

/tmp/ipykernel_6270/2614784211.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/home/sangam/genai/genai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [26]:
from youtube_transcript_api import YouTubeTranscriptApi

# Extract video ID from URL
video_id = "kS-CGkiPetQ"

try:
    api = YouTubeTranscriptApi()
    transcript_list = api.fetch(video_id, languages=['en'])
    
    # Combine all text parts into one string
    transcript_text = " ".join([item.text for item in transcript_list])
    
except Exception as e:
    print(f"✗ Error: {str(e)}")

In [27]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript_text])

In [28]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)

/home/sangam/genai/genai/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs={"k": 4})

In [31]:
retriever.invoke("How does google map work?")

[Document(id='2c94f878-bc93-49cb-b88e-ee6247f45a8d', metadata={}, page_content="method named after me. Dijkstra's algorithm is still regarded as one of the most elegant,\nshortest path algorithms. But it isn't good enough for Google Maps. Let's say I want to get from\nNewark Airport in New Jersey over to the Central Park Zoo. First, Dijkstra's algorithm checks all the 10-kilometer journeys, and then all the 20 kilometer journeys, and so on until it reaches\nall the 28 kilometer journeys, which includes the zoo. The search frontier\ncovers over 65,000 nodes, and includes places that are way off, like Staten Island, and\nlarge swaths of New Jersey. But even though it searched\nin a illogical directions, the runtime was about a 10th of a second, which is incredibly fast. And for anyone wanting to set up their own pathfinding tests, we were able to get a\nfeel for these algorithms using some Python scripts and actual maps from OpenStreetMaps. Solving problems using code like this is as muc

In [52]:
from os import getenv
api_token = getenv("HUGGINGFACEHUB_API_TOKEN")
llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",
    task="conversational",
    max_new_tokens=100,
    temperature=0.1,
    huggingfacehub_api_token=api_token,
)

model = ChatHuggingFace(llm=llm)

In [68]:
prompt = PromptTemplate(
    template = """You are a helpful assistant for answering questions about the content of a YouTube video.
      Use the following retrieved information to answer the question.
        If you don't know the answer, say you don't know.
        dont say based on the provided information, just start your answer
        your max token is 100, so be concise and to the point
        context = {context}
        question = {question}""",
        input_variables=["context", "question"]
)

In [69]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [70]:
question = "How does google map work?"
docs = retriever.invoke(question)
context = "\n".join([doc.page_content for doc in docs])

In [71]:
final_prompt = prompt.invoke({"context": context, "question": question})

In [72]:
def format_docs(docs):
    return "\n".join([doc.page_content for doc in docs])

In [73]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

parallel_chain = ({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

In [74]:
main_chain = parallel_chain | prompt | model | parser
result = main_chain.invoke("How does google map work?")
print(result)

Google Maps uses a routing algorithm called contraction hierarchies, which is a variation of Dijkstra's algorithm. It's designed to handle large graphs and find the shortest path by distance or travel time.
